In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/raw/application_train.csv')
print(df.shape)
df.head()

(307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
print(df['TARGET'].value_counts())
print(df['TARGET'].value_counts(normalize=True).round(3))

TARGET
0    282686
1     24825
Name: count, dtype: int64
TARGET
0    0.919
1    0.081
Name: proportion, dtype: float64


In [4]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
print(missing_pct.head(20))

COMMONAREA_MEDI             69.9
COMMONAREA_MODE             69.9
COMMONAREA_AVG              69.9
NONLIVINGAPARTMENTS_MODE    69.4
NONLIVINGAPARTMENTS_MEDI    69.4
NONLIVINGAPARTMENTS_AVG     69.4
FONDKAPREMONT_MODE          68.4
LIVINGAPARTMENTS_AVG        68.4
LIVINGAPARTMENTS_MEDI       68.4
LIVINGAPARTMENTS_MODE       68.4
FLOORSMIN_MEDI              67.8
FLOORSMIN_MODE              67.8
FLOORSMIN_AVG               67.8
YEARS_BUILD_MODE            66.5
YEARS_BUILD_MEDI            66.5
YEARS_BUILD_AVG             66.5
OWN_CAR_AGE                 66.0
LANDAREA_AVG                59.4
LANDAREA_MEDI               59.4
LANDAREA_MODE               59.4
dtype: float64


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,307511.0,278180.518577,102790.175348,100002.0,189145.5,278202.0,367142.5,456255.0
TARGET,307511.0,0.080729,0.272419,0.0,0.0,0.0,0.0,1.0
CNT_CHILDREN,307511.0,0.417052,0.722121,0.0,0.0,0.0,1.0,19.0
AMT_INCOME_TOTAL,307511.0,168797.919297,237123.146279,25650.0,112500.0,147150.0,202500.0,117000000.0
AMT_CREDIT,307511.0,599025.999706,402490.776996,45000.0,270000.0,513531.0,808650.0,4050000.0
...,...,...,...,...,...,...,...,...
AMT_REQ_CREDIT_BUREAU_DAY,265992.0,0.007000,0.110757,0.0,0.0,0.0,0.0,9.0
AMT_REQ_CREDIT_BUREAU_WEEK,265992.0,0.034362,0.204685,0.0,0.0,0.0,0.0,8.0
AMT_REQ_CREDIT_BUREAU_MON,265992.0,0.267395,0.916002,0.0,0.0,0.0,0.0,27.0
AMT_REQ_CREDIT_BUREAU_QRT,265992.0,0.265474,0.794056,0.0,0.0,0.0,0.0,261.0


In [6]:
cat_cols = df.select_dtypes('object').columns
for col in cat_cols:
    print(f"{col}: {df[col].nunique()} unique values")

NAME_CONTRACT_TYPE: 2 unique values
CODE_GENDER: 3 unique values
FLAG_OWN_CAR: 2 unique values
FLAG_OWN_REALTY: 2 unique values
NAME_TYPE_SUITE: 7 unique values
NAME_INCOME_TYPE: 8 unique values
NAME_EDUCATION_TYPE: 5 unique values
NAME_FAMILY_STATUS: 6 unique values
NAME_HOUSING_TYPE: 6 unique values
OCCUPATION_TYPE: 18 unique values
WEEKDAY_APPR_PROCESS_START: 7 unique values
ORGANIZATION_TYPE: 58 unique values
FONDKAPREMONT_MODE: 4 unique values
HOUSETYPE_MODE: 3 unique values
WALLSMATERIAL_MODE: 7 unique values
EMERGENCYSTATE_MODE: 2 unique values


C:\Users\valer\AppData\Local\Temp\ipykernel_84\161640505.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes('object').columns


In [7]:
print(df['CODE_GENDER'].value_counts())

CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# Encode categoricals
df_model = df.copy()
cat_cols = df_model.select_dtypes('object').columns
for col in cat_cols:
    df_model[col] = pd.Categorical(df_model[col]).codes

X = df_model.drop(['SK_ID_CURR', 'TARGET'], axis=1)
y = df_model['TARGET']

# 5-fold stratified CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(n_estimators=1000, random_state=42)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
    
    val_preds = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_preds)
    scores.append(auc)
    print(f"Fold {fold+1} AUC: {auc:.4f}")

print(f"\nMean AUC: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

C:\Users\valer\AppData\Local\Temp\ipykernel_84\3703691338.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_model.select_dtypes('object').columns


[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025506 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 11301
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 116
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.24736
Early stopping, best iteration is:
[134]	valid_0's binary_logloss: 0.247282
Fold 1 AUC: 0.7539
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.024935 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if m

In [9]:
bureau = pd.read_csv('../data/raw/bureau.csv')
print(bureau.shape)
bureau.head()

(1716428, 17)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [10]:
# Encode categoricals
bureau_cat = ['CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE']
for col in bureau_cat:
    bureau[col] = pd.Categorical(bureau[col]).codes

# Aggregate to one row per applicant
bureau_agg = bureau.drop('SK_ID_BUREAU', axis=1).groupby('SK_ID_CURR').agg(
    ['mean', 'max', 'min', 'std', 'count']
)

# Flatten column names
bureau_agg.columns = ['BUREAU_' + '_'.join(col).upper() for col in bureau_agg.columns]
bureau_agg = bureau_agg.reset_index()

print(bureau_agg.shape)
print(bureau_agg.columns.tolist()[:10])

(305811, 76)
['SK_ID_CURR', 'BUREAU_CREDIT_ACTIVE_MEAN', 'BUREAU_CREDIT_ACTIVE_MAX', 'BUREAU_CREDIT_ACTIVE_MIN', 'BUREAU_CREDIT_ACTIVE_STD', 'BUREAU_CREDIT_ACTIVE_COUNT', 'BUREAU_CREDIT_CURRENCY_MEAN', 'BUREAU_CREDIT_CURRENCY_MAX', 'BUREAU_CREDIT_CURRENCY_MIN', 'BUREAU_CREDIT_CURRENCY_STD']


In [11]:
df = df.merge(bureau_agg, on='SK_ID_CURR', how='left')
print(df.shape)

(307511, 197)


In [12]:
previous_application = pd.read_csv('../data/raw/previous_application.csv')
print(previous_application.shape)
previous_application.head()

(1670214, 37)


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
previous_application.select_dtypes('object').columns.tolist()

C:\Users\valer\AppData\Local\Temp\ipykernel_84\2179804641.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  previous_application.select_dtypes('object').columns.tolist()


['WEEKDAY_APPR_PROCESS_START',
 'FLAG_LAST_APPL_PER_CONTRACT',
 'NAME_CASH_LOAN_PURPOSE',
 'NAME_CONTRACT_STATUS',
 'NAME_PAYMENT_TYPE',
 'CODE_REJECT_REASON',
 'NAME_TYPE_SUITE',
 'NAME_CLIENT_TYPE',
 'NAME_GOODS_CATEGORY',
 'NAME_PORTFOLIO',
 'NAME_PRODUCT_TYPE',
 'CHANNEL_TYPE',
 'NAME_SELLER_INDUSTRY',
 'NAME_YIELD_GROUP',
 'PRODUCT_COMBINATION']

In [19]:
previous_application_cat = previous_application.select_dtypes('object').columns.tolist()
for col in previous_application_cat:
    previous_application[col] = pd.Categorical(previous_application[col]).codes

previous_application_agg = previous_application.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(
    ['mean', 'max', 'min', 'std', 'count']
)

previous_application_agg.columns = ['PREV_' + '_'.join(col).upper() for col in previous_application_agg.columns]
previous_application_agg = previous_application_agg.reset_index()

df = df.merge(previous_application_agg, on='SK_ID_CURR', how='left')
print(df.shape)

C:\Users\valer\AppData\Local\Temp\ipykernel_84\3153011151.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  previous_application_cat = previous_application.select_dtypes('object').columns.tolist()


(307511, 372)


In [20]:
# POS_CASH_balance
pos_cash = pd.read_csv('../data/raw/POS_CASH_balance.csv')
pos_cash_cat = pos_cash.select_dtypes('object').columns.tolist()
for col in pos_cash_cat:
    pos_cash[col] = pd.Categorical(pos_cash[col]).codes
pos_cash_agg = pos_cash.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
pos_cash_agg.columns = ['POS_' + '_'.join(col).upper() for col in pos_cash_agg.columns]
pos_cash_agg = pos_cash_agg.reset_index()
df = df.merge(pos_cash_agg, on='SK_ID_CURR', how='left')
print(df.shape)

C:\Users\valer\AppData\Local\Temp\ipykernel_84\2169182274.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  pos_cash_cat = pos_cash.select_dtypes('object').columns.tolist()


(307511, 402)


In [21]:
# credit_card_balance
credit_card = pd.read_csv('../data/raw/credit_card_balance.csv')
credit_card_cat = credit_card.select_dtypes('object').columns.tolist()
for col in credit_card_cat:
    credit_card[col] = pd.Categorical(credit_card[col]).codes
credit_card_agg = credit_card.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
credit_card_agg.columns = ['CC_' + '_'.join(col).upper() for col in credit_card_agg.columns]
credit_card_agg = credit_card_agg.reset_index()
df = df.merge(credit_card_agg, on='SK_ID_CURR', how='left')
print(df.shape)

C:\Users\valer\AppData\Local\Temp\ipykernel_84\4123326327.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  credit_card_cat = credit_card.select_dtypes('object').columns.tolist()


(307511, 507)


In [22]:
# installments_payments
installments = pd.read_csv('../data/raw/installments_payments.csv')
installments_cat = installments.select_dtypes('object').columns.tolist()
for col in installments_cat:
    installments[col] = pd.Categorical(installments[col]).codes
installments_agg = installments.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
installments_agg.columns = ['INST_' + '_'.join(col).upper() for col in installments_agg.columns]
installments_agg = installments_agg.reset_index()
df = df.merge(installments_agg, on='SK_ID_CURR', how='left')
print(df.shape)

(307511, 537)


In [23]:
bureau_balance = pd.read_csv('../data/raw/bureau_balance.csv')
print(bureau_balance.shape)
bureau_balance.head()

(27299925, 3)


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [24]:
# Aggregate bureau_balance by SK_ID_BUREAU
bureau_balance_cat = bureau_balance.select_dtypes('object').columns.tolist()
for col in bureau_balance_cat:
    bureau_balance[col] = pd.Categorical(bureau_balance[col]).codes

bb_agg = bureau_balance.groupby('SK_ID_BUREAU').agg(['mean', 'max', 'min', 'std', 'count'])
bb_agg.columns = ['BB_' + '_'.join(col).upper() for col in bb_agg.columns]
bb_agg = bb_agg.reset_index()

# Merge into bureau, then aggregate by SK_ID_CURR
bureau_full = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
bureau_full_agg = bureau_full.drop('SK_ID_BUREAU', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
bureau_full_agg.columns = ['BUREAU_BB_' + '_'.join(col).upper() for col in bureau_full_agg.columns]
bureau_full_agg = bureau_full_agg.reset_index()

# Merge into main table
df = df.merge(bureau_full_agg, on='SK_ID_CURR', how='left')
print(df.shape)

C:\Users\valer\AppData\Local\Temp\ipykernel_84\2103427313.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  bureau_balance_cat = bureau_balance.select_dtypes('object').columns.tolist()


(307511, 662)


In [25]:
# Encode remaining categoricals in main table
cat_cols = df.select_dtypes('object').columns.tolist()
for col in cat_cols:
    df[col] = pd.Categorical(df[col]).codes

X = df.drop(['SK_ID_CURR', 'TARGET'], axis=1)
y = df['TARGET']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(n_estimators=1000, random_state=42)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
    
    val_preds = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_preds)
    scores.append(auc)
    print(f"Fold {fold+1} AUC: {auc:.4f}")

print(f"\nMean AUC: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

C:\Users\valer\AppData\Local\Temp\ipykernel_84\23640947.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes('object').columns.tolist()


[LightGBM] [Info] Number of positive: 19860, number of negative: 226148
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.085000 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97987
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 656
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432482
[LightGBM] [Info] Start training from score -2.432482
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.24102
[200]	valid_0's binary_logloss: 0.24091
Early stopping, best iteration is:
[173]	valid_0's binary_logloss: 0.240769
Fold 1 AUC: 0.7762
[LightGBM] [Info] Number of positive: 19860, number of negative: 226149
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.140307 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you

In [26]:
import optuna

def objective(trial):
    params = {
        'n_estimators': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'random_state': 42
    }
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(-1)])
        
        val_preds = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, val_preds))
    
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

print(f"Best AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

c:\Users\valer\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-05-28 11:00:59,174] A new study created in memory with name: no-name-9d01495e-7f34-4fa0-957c-b37335b70a46


[LightGBM] [Info] Number of positive: 16550, number of negative: 188457
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.933183 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 98063
[LightGBM] [Info] Number of data points in the train set: 205007, number of used features: 652
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432484
[LightGBM] [Info] Start training from score -2.432484
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

[I 2026-05-28 11:03:12,344] Trial 0 finished with value: 0.7793264327528652 and parameters: {'learning_rate': 0.0673695744906154, 'num_leaves': 84, 'max_depth': 11, 'min_child_samples': 96, 'subsample': 0.7011370497970648, 'colsample_bytree': 0.9340698235544067, 'reg_alpha': 0.051743331462548836, 'reg_lambda': 0.47386800810404994}. Best is trial 0 with value: 0.7793264327528652.


[LightGBM] [Info] Number of positive: 16550, number of negative: 188457
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.969370 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 98063
[LightGBM] [Info] Number of data points in the train set: 205007, number of used features: 652
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432484
[LightGBM] [Info] Start training from score -2.432484
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Early stopping, best iteration is:
[364]	valid_0's binary_logloss: 0.238442
[LightGBM] [Info] Number of positive: 16550, number of negative: 

[I 2026-05-28 11:05:42,496] Trial 1 finished with value: 0.782289535647236 and parameters: {'learning_rate': 0.0475003875567915, 'num_leaves': 36, 'max_depth': 8, 'min_child_samples': 77, 'subsample': 0.9309395133893187, 'colsample_bytree': 0.7744558181459309, 'reg_alpha': 0.41206808867749234, 'reg_lambda': 0.48770678591546535}. Best is trial 1 with value: 0.782289535647236.


...

[LightGBM] [Info] Number of positive: 16550, number of negative: 188457
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.109816 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 98069
[LightGBM] [Info] Number of data points in the train set: 205007, number of used features: 655
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432484
[LightGBM] [Info] Start training from score -2.432484
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[L

[I 2026-05-28 11:50:21,792] Trial 18 finished with value: 0.7832338642670411 and parameters: {'learning_rate': 0.08158747028756702, 'num_leaves': 53, 'max_depth': 4, 'min_child_samples': 21, 'subsample': 0.6313075989367224, 'colsample_bytree': 0.9379804429199674, 'reg_alpha': 0.48111996392531214, 'reg_lambda': 0.6379361068494286}. Best is trial 11 with value: 0.7838257275518504.


[LightGBM] [Info] Number of positive: 16550, number of negative: 188457
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.910169 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 98069
[LightGBM] [Info] Number of data points in the train set: 205007, number of used features: 655
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432484
[LightGBM] [Info] Start training from score -2.432484
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

[I 2026-05-28 11:52:41,389] Trial 19 finished with value: 0.7834123611454 and parameters: {'learning_rate': 0.06485126621630449, 'num_leaves': 96, 'max_depth': 3, 'min_child_samples': 35, 'subsample': 0.6773407393002636, 'colsample_bytree': 0.8372884170790522, 'reg_alpha': 0.7490466760428849, 'reg_lambda': 0.03649636203604434}. Best is trial 11 with value: 0.7838257275518504.


Best AUC: 0.7838
Best params: {'learning_rate': 0.09866949640178968, 'num_leaves': 68, 'max_depth': 3, 'min_child_samples': 43, 'subsample': 0.6028140604448102, 'colsample_bytree': 0.8713900411046183, 'reg_alpha': 0.5544275564463821, 'reg_lambda': 0.708285789751911}


In [28]:
best_params = study.best_params
best_params['n_estimators'] = 1000
best_params['random_state'] = 42

# Train on full training data
final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X, y, callbacks=[lgb.log_evaluation(100)])

# Load and prepare test data
test = pd.read_csv('../data/raw/application_test.csv')
test_ids = test['SK_ID_CURR']

[LightGBM] [Info] Number of positive: 24825, number of negative: 282686
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.893485 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 97953
[LightGBM] [Info] Number of data points in the train set: 307511, number of used features: 655
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080729 -> initscore=-2.432486
[LightGBM] [Info] Start training from score -2.432486
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

In [29]:
# Encode categoricals
bureau_cat = bureau.select_dtypes('object').columns.tolist()
for col in bureau_cat:
    bureau[col] = pd.Categorical(bureau[col]).codes

# Aggregate to one row per applicant
bureau_agg = bureau.drop('SK_ID_BUREAU', axis=1).groupby('SK_ID_CURR').agg(
    ['mean', 'max', 'min', 'std', 'count']
)

# Flatten column names
bureau_agg.columns = ['BUREAU_' + '_'.join(col).upper() for col in bureau_agg.columns]
bureau_agg = bureau_agg.reset_index()

print(bureau_agg.shape)
print(bureau_agg.columns.tolist()[:10])

test = test.merge(bureau_agg, on='SK_ID_CURR', how='left')
print(test.shape)

(305811, 76)
['SK_ID_CURR', 'BUREAU_CREDIT_ACTIVE_MEAN', 'BUREAU_CREDIT_ACTIVE_MAX', 'BUREAU_CREDIT_ACTIVE_MIN', 'BUREAU_CREDIT_ACTIVE_STD', 'BUREAU_CREDIT_ACTIVE_COUNT', 'BUREAU_CREDIT_CURRENCY_MEAN', 'BUREAU_CREDIT_CURRENCY_MAX', 'BUREAU_CREDIT_CURRENCY_MIN', 'BUREAU_CREDIT_CURRENCY_STD']
(48744, 196)


In [30]:
previous_application_cat = previous_application.select_dtypes('object').columns.tolist()
for col in previous_application_cat:
    previous_application[col] = pd.Categorical(previous_application[col]).codes

previous_application_agg = previous_application.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(
    ['mean', 'max', 'min', 'std', 'count']
)

previous_application_agg.columns = ['PREV_' + '_'.join(col).upper() for col in previous_application_agg.columns]
previous_application_agg = previous_application_agg.reset_index()

test = test.merge(previous_application_agg, on='SK_ID_CURR', how='left')
print(test.shape)

(48744, 371)


In [31]:
# POS_CASH_balance
pos_cash = pd.read_csv('../data/raw/POS_CASH_balance.csv')
pos_cash_cat = pos_cash.select_dtypes('object').columns.tolist()
for col in pos_cash_cat:
    pos_cash[col] = pd.Categorical(pos_cash[col]).codes
pos_cash_agg = pos_cash.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
pos_cash_agg.columns = ['POS_' + '_'.join(col).upper() for col in pos_cash_agg.columns]
pos_cash_agg = pos_cash_agg.reset_index()
test = test.merge(pos_cash_agg, on='SK_ID_CURR', how='left')
print(test.shape)

C:\Users\valer\AppData\Local\Temp\ipykernel_84\4138451004.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  pos_cash_cat = pos_cash.select_dtypes('object').columns.tolist()


(48744, 401)


In [32]:
# credit_card_balance
credit_card = pd.read_csv('../data/raw/credit_card_balance.csv')
credit_card_cat = credit_card.select_dtypes('object').columns.tolist()
for col in credit_card_cat:
    credit_card[col] = pd.Categorical(credit_card[col]).codes
credit_card_agg = credit_card.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
credit_card_agg.columns = ['CC_' + '_'.join(col).upper() for col in credit_card_agg.columns]
credit_card_agg = credit_card_agg.reset_index()
test = test.merge(credit_card_agg, on='SK_ID_CURR', how='left')
print(test.shape)

C:\Users\valer\AppData\Local\Temp\ipykernel_84\624323345.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  credit_card_cat = credit_card.select_dtypes('object').columns.tolist()


(48744, 506)


In [33]:
# installments_payments
installments = pd.read_csv('../data/raw/installments_payments.csv')
installments_cat = installments.select_dtypes('object').columns.tolist()
for col in installments_cat:
    installments[col] = pd.Categorical(installments[col]).codes
installments_agg = installments.drop('SK_ID_PREV', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
installments_agg.columns = ['INST_' + '_'.join(col).upper() for col in installments_agg.columns]
installments_agg = installments_agg.reset_index()
test = test.merge(installments_agg, on='SK_ID_CURR', how='left')
print(test.shape)

(48744, 536)


In [34]:
# Aggregate bureau_balance by SK_ID_BUREAU
bureau_balance_cat = bureau_balance.select_dtypes('object').columns.tolist()
for col in bureau_balance_cat:
    bureau_balance[col] = pd.Categorical(bureau_balance[col]).codes

bb_agg = bureau_balance.groupby('SK_ID_BUREAU').agg(['mean', 'max', 'min', 'std', 'count'])
bb_agg.columns = ['BB_' + '_'.join(col).upper() for col in bb_agg.columns]
bb_agg = bb_agg.reset_index()

# Merge into bureau, then aggregate by SK_ID_CURR
bureau_full = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')
bureau_full_agg = bureau_full.drop('SK_ID_BUREAU', axis=1).groupby('SK_ID_CURR').agg(['mean', 'max', 'min', 'std', 'count'])
bureau_full_agg.columns = ['BUREAU_BB_' + '_'.join(col).upper() for col in bureau_full_agg.columns]
bureau_full_agg = bureau_full_agg.reset_index()

# Merge into main table
test = test.merge(bureau_full_agg, on='SK_ID_CURR', how='left')
print(test.shape)

(48744, 661)


In [35]:
# Encode categoricals
cat_cols = test.select_dtypes('object').columns.tolist()
for col in cat_cols:
    test[col] = pd.Categorical(test[col]).codes

# Drop SK_ID_CURR, no TARGET to drop
X_test = test.drop(['SK_ID_CURR'], axis=1)

# Predict
test_preds = final_model.predict_proba(X_test)[:, 1]

# Save submission
submission = pd.DataFrame({'SK_ID_CURR': test_ids, 'TARGET': test_preds})
submission.to_csv('../outputs/submission.csv', index=False)
print(submission.head())

C:\Users\valer\AppData\Local\Temp\ipykernel_84\2088919260.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = test.select_dtypes('object').columns.tolist()


   SK_ID_CURR    TARGET
0      100001  0.032255
1      100005  0.131011
2      100013  0.024432
3      100028  0.035979
4      100038  0.120429
